# CS 3892 / 5892 — Session 17 · Bounded model checking of programs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-27-bounded-model-checking-of-programs.ipynb)

**Tuesday, October 27, 2026.** A C program is a transition system over its memory states. **CBMC**
unrolls its loops up to a bound, turns the unrolled program into one formula, and asks a SAT solver
whether any assertion can fail: bounded model checking, as in session 16, with the program itself as
the model. Four programs: the running counter, session 11's agent, a retry loop that shows what an
**unwinding assertion** is for, and Euclid's GCD, with a specification too weak to notice a bug --
then the same GCD loop in NuSMV.

The bounded and inductive checks on SMV models from the first half of class are in the
[session 16 notebook](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-20-symbolic-and-bounded-model-checking.ipynb).
Every file here asserts its own results. CBMC (free, BSD) and NuSMV (free, LGPL) are fetched by the setup cells.

## Setup

Run these three once.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, fetch CBMC, define helpers
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-27-bounded-model-checking-of-programs"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

# Colab does not ship Z3; install it if this machine lacks it.
try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

PYD = ROOT / "sessions" / SESSION / "python"
CD  = ROOT / "sessions" / SESSION / "c"
FIG = ROOT / "sessions" / SESSION / "figures"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True, cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

from IPython.display import SVG, display
print("ready — helpers: show(path), run(path)")

In [ ]:
# --- CBMC: the official 6.11.0 release, fetched once (about 70 MB) and checked by SHA-256.
# cbmc_cli.py unpacks the Ubuntu 22.04 package in Python: no root, no dpkg needed.
out = run(PYD / "cbmc_cli.py")
assert "CBMC 6.11.0" in out

In [ ]:
# --- NuSMV, for the SMV model in section 4 (same cell as sessions 9-16) -------
# NuSMV is free (LGPL) from nusmv.fbk.eu. 2.7.1 is tried first; it needs glibc >= 2.38
# and libedit, which an older Colab image may lack, so the static 2.6.0 build is the
# fallback. Same verdicts. (nuXmv, its successor, gives the same verdicts on this file.)
import shutil
SMV = ROOT / "sessions" / SESSION / "smv"

# Each archive is pinned by SHA-256 (taken Oct 30 2026; FBK publishes none). A download that does
# not match is refused and never unpacked or run.
BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz",
           "894dc71f0997683f28d4dc3512e3a6b608e02b9700e5e813017931df2d11883c"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",
           "11b12e42bc04566c4ec1f7cc205fa8704382ac4d69e052154cbb59c9b7e8e08c")]

def _fetch(url, sha, dest):
    """Download; refuse unless the SHA-256 matches the pin; unpack with the top folder stripped."""
    import hashlib, tarfile, tempfile, urllib.request
    with tempfile.TemporaryDirectory() as tmp:
        arc = pathlib.Path(tmp) / url.rsplit("/", 1)[1]
        with urllib.request.urlopen(url, timeout=120) as r, open(arc, "wb") as fh:
            shutil.copyfileobj(r, fh)
        got = hashlib.sha256(arc.read_bytes()).hexdigest()
        if got != sha:
            raise RuntimeError(f"REFUSING {url}: SHA-256 {got} is not the pinned {sha}")
        with tarfile.open(arc) as tar:
            for m in tar.getmembers():
                parts = pathlib.Path(m.name).parts[1:]
                if not parts or m.issym() or m.islnk() or ".." in parts:
                    continue
                m.name = str(pathlib.Path(*parts))
                tar.extract(m, dest)

def _works(exe):
    """True only if this binary actually model-checks a file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "04_gcd.smv")], capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, sha in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        _fetch(url, sha, dest)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def verdicts(path):
    """Run NuSMV on a file; print and return {formula: verdict}, plus the full output.
    NuSMV does not print verdicts in file order, so match them by formula."""
    out = subprocess.run([NUSMV, str(path)], capture_output=True, text=True).stdout
    res = {}
    for line in out.splitlines():
        m = re.match(r"-- (?:specification|invariant)\s+(.*?)\s+is (true|false)$", line)
        if m:
            res[re.sub(r"\s+", " ", m.group(1)).strip()] = m.group(2)
    for k, v in res.items():
        print(f"{v:6} {k}")
    return res, out

## 1. A program is a transition system

The running counter, as a loop. A state is (the line the program is at, the value of `x`).
CBMC checks `x <= 10` after every one of 30 steps. Then the formula CBMC builds for it at bound 2:
`x_2 = 0` is I, each copy of the loop body becomes equations with Boolean guards (both branches of
the `if`, and a guard picks one), and the property comes last. That is session 16's BMC formula,
with the program as T.

In [ ]:
display(SVG(filename=str(FIG / "counter.svg")))
show(CD / "counter.c")
out = run(PYD / "01_counter_in_c.py")
assert "VERIFICATION SUCCESSFUL" in out
assert "x_5 = (g1 ∧ g2 ? x_3 : x_4)" in out and "(g1 ∧ g3) ⇒ (x_8 ≤ 10)" in out

## 2. Reading a counterexample: the agent, in C

`nondet_int()` has no body, so CBMC may make it return anything: the model's choice of action, and
the human's answer. The trace is the list of values that makes the assertion fail.

In [ ]:
display(SVG(filename=str(FIG / "agent_c.svg")))
show(CD / "agent.c")
out = run(PYD / "02_agent_trace.py")
assert "assertion approved: FAILURE" in out and "VERIFICATION SUCCESSFUL" in out

## 3. What an unwinding assertion protects you from

`--unwind k` copies each loop body *k* times. A run that leaves the loop has at most *k* − 1
iterations; the *k*-th copy is where CBMC checks that the loop has stopped. That check is the **unwinding assertion**
(on by default since CBMC 6) checks that no run is longer than that. Turn it off and a too-small
bound gives a confident, wrong answer.

In [ ]:
display(SVG(filename=str(FIG / "retry.svg")))
show(CD / "retry.c")
out = run(PYD / "03_unwinding_assertions.py")
assert "unwinding assertion loop 0: FAILURE" in out and "counterexample: max_retries" in out

## 4. Euclid's GCD: a specification can be too weak

Euclid's algorithm by subtraction, for every `a0, b0` from 1 to 20. "g divides both" is a true
claim -- and the broken version, which returns 1, satisfies it too. Only "no larger number divides
both" tells them apart. That is D7's question in one line, and HW3 part 5's.

In [ ]:
display(SVG(filename=str(FIG / "gcd.svg")))
show(CD / "gcd.c")
out = run(PYD / "04_gcd.py")
assert "VERIFICATION SUCCESSFUL" in out and "SUCCESSFUL, and wrong" in out
assert "goes round Euclid's loop 19 times" in out and "** 0 of 8 failed" in out
assert "a0 = 20, b0 = 20, gcd returns g = 1, then d = 2 divides both" in out

### The same loop in SMV: every input pair, every run

`a0`, `b0` and `d` are `FROZENVAR`s: chosen at the start, never changed. NuSMV checks every initial
state, so the third property holds for **every** `d` -- "no larger number divides both", without a
quantifier. The last property is false, and the counterexample is a lasso: a pair whose gcd is not
1, sitting at `a = b` forever. (nuXmv gives the same four verdicts.)

In [ ]:
show(SMV / "04_gcd.smv")
res, out = verdicts(SMV / "04_gcd.smv")
assert res["F a = b"] == "true" and res["F a = 1"] == "false"
assert res["(a = b -> (a0 mod a = 0 & b0 mod a = 0))"] == "true"
assert res["(((a = b & a0 mod d = 0) & b0 mod d = 0) -> d <= a)"] == "true"
assert "Loop starts here" in out                     # the counterexample to F (a = 1) is a lasso

## 5. What IC3's frames are

A **frame** Fr_i is a set that contains every state reachable in at most i steps: an
over-approximation. IC3 keeps Fr_0 = I, Fr_1, ..., Fr_k with one step from a frame landing in the
next, and no bad state in any frame. It shrinks a frame when it shows a state cannot be reached
within i steps, first excluding the state's predecessor one frame down. When two neighbouring
frames are equal, that frame is an inductive invariant.

This small IC3 excludes one state at a time on session 16's finite add-2 (x in 0..15, P: x != 5)
and prints every frame after every round. Real IC3 learns clauses covering many states at once.

In [ ]:
display(SVG(filename=str(FIG / "add2_mod16.svg")))
out = run(PYD / "05_ic3_frames.py")
assert "round 1: Fr_1 = all but {5}" in out
assert "Fr_1 = Fr_2 = [0, 2, 4, 6, 8, 10, 12, 14]" in out

## Try it yourself

- In `retry.c`, change the claim to `attempts <= 5`. Which bound is the smallest that proves it, and
  what does CBMC say at the bound below that?
- In `agent.c`, delete the line `approved = 0;` after a delete. Does the property still hold with
  `-DGATEWAY`? What does that say about one approval and several deletes?
- Make the counter's cap a `nondet_int()` between 1 and 20. Which `--unwind` do you need now?
- In `gcd.c`, lower `--unwind` until the unwinding assertion fails. Which inputs does the trace
  pick, and why those?
- In `04_gcd.smv`, drop `a = b &` from the third property. Is it still true? What does that say about
  every state of the loop, not only the last one? (Thursday's word for it: an invariant.)